<a href="https://colab.research.google.com/github/Pedro4Albuquerque/Processamento-de-Linguagem-Natural/blob/main/Atividade%20de%20Web%20Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## **Aluno:** Pedro Henrique Albuquerque Souza

**Ferramenta de apoio:** ChatGPT, utilizado como ferramenta de consulta e auxílio no desenvolvimento da atividade.

**Objetivo:** Desenvolver um script Python para coletar automaticamente informações de livros do site Books to Scrape, incluindo nome, preço e disponibilidade, armazenando os dados em um banco SQLite.

### **Prompt**:
"Olá! Preciso de ajuda para desenvolver uma atividade prática de Web Scraping utilizando Python no Google Colab.

O objetivo é criar um script que colete automaticamente informações de livros do site de testes https://books.toscrape.com/ e armazene os dados em um banco de dados SQLite.

Preciso que o código realize as seguintes etapas:

Acesse o site e percorra automaticamente todas as páginas disponíveis, coletando os dados dos livros.

Extraia as seguintes informações de cada livro:

Nome do livro;

Preço em libras esterlinas (£);

Disponibilidade em estoque;

URL do livro;

Data e hora da coleta.

Crie um banco de dados SQLite chamado livros.db, com uma tabela para armazenar as informações coletadas, utilizando um identificador único para cada livro e evitando registros duplicados.

Utilize as bibliotecas requests e BeautifulSoup para acessar as páginas e extrair as informações do HTML.

Utilize a biblioteca pandas para consultar os registros armazenados no banco e apresentar os resultados em formato de tabela.

Ao finalizar a coleta, apresente a quantidade de páginas percorridas, o total de livros coletados e a quantidade de registros armazenados no banco.

Implemente uma validação para verificar a estrutura do banco de dados e identificar possíveis valores nulos nos registros coletados.

Organize o código em etapas, com comentários simples e explicativos, para facilitar o entendimento de quem está aprendendo Python e Web Scraping."

# Atividade de Web Scraping

In [1]:
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

URL_INICIAL = "https://books.toscrape.com/"
ARQUIVO_BANCO = "livros.db"
ESPERA_ENTRE_PAGINAS = 0.3  # segundos; evita requisicoes muito rapidas


def extrair_livros(html: str, url_pagina: str):
    """Lê uma página HTML e devolve os livros e o endereço da próxima página."""
    pagina = BeautifulSoup(html, "html.parser")
    livros = []

    for card in pagina.select("article.product_pod"):
        link = card.select_one("h3 a")
        elemento_preco = card.select_one("p.price_color")
        elemento_estoque = card.select_one("p.availability")

        if not (link and elemento_preco and elemento_estoque):
            print("Aviso: livro com campos ausentes; registro ignorado.")
            continue

        nome = link.get("title") or link.get_text(" ", strip=True)
        preco_texto = elemento_preco.get_text(strip=True)
        preco_gbp = float(preco_texto.replace("£", "").replace(",", "."))
        estoque_texto = elemento_estoque.get_text(" ", strip=True).lower()
        disponibilidade = "Disponível" if "in stock" in estoque_texto else "Indisponível"
        url_livro = urljoin(url_pagina, link["href"])

        livros.append((nome, preco_gbp, disponibilidade, url_livro))

    proxima = pagina.select_one("li.next a")
    url_proxima = urljoin(url_pagina, proxima["href"]) if proxima else None
    return livros, url_proxima


def coletar_catalogo():
    """Visita todas as páginas do catálogo e reúne seus livros."""
    url = URL_INICIAL
    numero_pagina = 0
    todos_os_livros = []

    with requests.Session() as sessao:
        sessao.headers.update({"User-Agent": "AtividadeEducacionalWebScraping/1.0"})

        while url:
            resposta = sessao.get(url, timeout=30)
            resposta.raise_for_status()  # apresenta o erro, se a página não carregar
            resposta.encoding = "utf-8"

            livros, url = extrair_livros(resposta.text, resposta.url)
            if not livros:
                raise RuntimeError("Nenhum livro encontrado. Verifique o HTML do site.")

            numero_pagina += 1
            todos_os_livros.extend(livros)
            print(f"Página {numero_pagina:02d}: {len(livros)} livros coletados")

            if url:
                time.sleep(ESPERA_ENTRE_PAGINAS)

    print(f"\nTotal encontrado: {len(todos_os_livros)} livros em {numero_pagina} páginas.")
    return todos_os_livros


def salvar_no_sqlite(livros, arquivo_banco=ARQUIVO_BANCO):
    """Cria a tabela e insere ou atualiza os livros, sem duplicar URLs."""
    data_coleta = datetime.now(timezone.utc).isoformat(timespec="seconds")
    registros = [(*livro, data_coleta) for livro in livros]

    with sqlite3.connect(arquivo_banco) as conexao:
        conexao.execute("""
            CREATE TABLE IF NOT EXISTS livros (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                preco_gbp REAL NOT NULL,
                disponibilidade TEXT NOT NULL,
                url TEXT NOT NULL UNIQUE,
                coletado_em TEXT NOT NULL
            )
        """)
        conexao.executemany("""
            INSERT INTO livros (nome, preco_gbp, disponibilidade, url, coletado_em)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(url) DO UPDATE SET
                nome = excluded.nome,
                preco_gbp = excluded.preco_gbp,
                disponibilidade = excluded.disponibilidade,
                coletado_em = excluded.coletado_em
        """, registros)
        total = conexao.execute("SELECT COUNT(*) FROM livros").fetchone()[0]

    print(f"Banco salvo em: {Path(arquivo_banco).resolve()}")
    print(f"Registros atualmente no banco: {total}")
    return total


def main():
    livros = coletar_catalogo()
    salvar_no_sqlite(livros)

    with sqlite3.connect(ARQUIVO_BANCO) as conexao:
        print("\nExemplo de consulta (5 primeiros livros):")
        for nome, preco, disponibilidade in conexao.execute("""
            SELECT nome, preco_gbp, disponibilidade
            FROM livros ORDER BY id LIMIT 5
        """):
            print(f"{nome} | £{preco:.2f} | {disponibilidade}")


if __name__ == "__main__":
    main()

Página 01: 20 livros coletados
Página 02: 20 livros coletados
Página 03: 20 livros coletados
Página 04: 20 livros coletados
Página 05: 20 livros coletados
Página 06: 20 livros coletados
Página 07: 20 livros coletados
Página 08: 20 livros coletados
Página 09: 20 livros coletados
Página 10: 20 livros coletados
Página 11: 20 livros coletados
Página 12: 20 livros coletados
Página 13: 20 livros coletados
Página 14: 20 livros coletados
Página 15: 20 livros coletados
Página 16: 20 livros coletados
Página 17: 20 livros coletados
Página 18: 20 livros coletados
Página 19: 20 livros coletados
Página 20: 20 livros coletados
Página 21: 20 livros coletados
Página 22: 20 livros coletados
Página 23: 20 livros coletados
Página 24: 20 livros coletados
Página 25: 20 livros coletados
Página 26: 20 livros coletados
Página 27: 20 livros coletados
Página 28: 20 livros coletados
Página 29: 20 livros coletados
Página 30: 20 livros coletados
Página 31: 20 livros coletados
Página 32: 20 livros coletados
Página 3

In [2]:
import sqlite3

conn = sqlite3.connect("/content/livros.db")

print("Tabelas:", conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall())

conn.close()

Tabelas: [('livros',), ('sqlite_sequence',)]


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/content/livros.db")

# Carregar os livros
df = pd.read_sql_query("SELECT * FROM livros", conn)

print("Total de livros:", len(df))
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nPrimeiros 10 livros:")
display(df.head(10))

conn.close()

Total de livros: 1000

Colunas disponíveis:
['id', 'nome', 'preco_gbp', 'disponibilidade', 'url', 'coletado_em']

Valores ausentes:
id                 0
nome               0
preco_gbp          0
disponibilidade    0
url                0
coletado_em        0
dtype: int64

Primeiros 10 livros:


,id,nome,preco_gbp,disponibilidade,url,coletado_em
0,1,A Light in the Attic,51.77,Disponível,https://books.toscrape.com/catalogue/a-light-i...,2026-10-21T23:05:40+00:00
1,2,Tipping the Velvet,53.74,Disponível,https://books.toscrape.com/catalogue/tipping-t...,2026-10-21T23:05:40+00:00
2,3,Soumission,50.10,Disponível,https://books.toscrape.com/catalogue/soumissio...,2026-10-21T23:05:40+00:00
3,4,Sharp Objects,47.82,Disponível,https://books.toscrape.com/catalogue/sharp-obj...,2026-10-21T23:05:40+00:00
4,5,Sapiens: A Brief History of Humankind,54.23,Disponível,https://books.toscrape.com/catalogue/sapiens-a...,2026-10-21T23:05:40+00:00
5,6,The Requiem Red,22.65,Disponível,https://books.toscrape.com/catalogue/the-requi...,2026-10-21T23:05:40+00:00
6,7,The Dirty Little Secrets of Getting Your Dream...,33.34,Disponível,https://books.toscrape.com/catalogue/the-dirty...,2026-10-21T23:05:40+00:00
7,8,The Coming Woman: A Novel Based on the Life of...,17.93,Disponível,https://books.toscrape.com/catalogue/the-comin...,2026-10-21T23:05:40+00:00
8,9,The Boys in the Boat: Nine Americans and Their...,22.60,Disponível,https://books.toscrape.com/catalogue/the-boys-...,2026-10-21T23:05:40+00:00
9,10,The Black Maria,52.15,Disponível,https://books.toscrape.com/catalogue/the-black...,2026-10-21T23:05:40+00:00
